In [1]:
from pathlib import Path
import os, sys, io, json, zipfile, shutil, subprocess, random, math
from datetime import datetime

# =========================================================
# INSTALLS
# =========================================================
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "pillow", "matplotlib", "tqdm"],
    check=True
)

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# =========================================================
# SETTINGS
# =========================================================
MODE = "train"                 # "train" or "status"
EXPERIMENT_MODE = "freq_only"  # "freq_only" or "combined"

# Fast ablation only
AB_KIMG = 3
AB_TICK = 1
AB_SNAP = 1
AB_METRICS = "none"
AB_SEED = 42

# Keep same safe GAN settings as working runs
CFG = "stylegan3-r"
GAMMA = 1.0
CBASE = 16384
GLR = 0.0025
DLR = 0.0025
AUG = "noaug"
MIRROR = 0
BATCH = 2
BATCH_GPU = 1
WORKERS = 1
MBSTD_GROUP = 1

# Landmark settings (only used if EXPERIMENT_MODE == "combined")
LM_LAMBDA = 0.25
LM_Z_MARGIN = 2.5

# Frequency ablation settings
FREQ_SAMPLE_SIZE = 3000
FREQ_CUTOFF_RATIO = 0.25
FREQ_LAMBDA = 0.15
FREQ_Z_MARGIN = 1.0

# =========================================================
# PATHS
# =========================================================
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

dataset_zip = ROOT / "data" / "stylegan3_zip" / "celeba_256x256.zip"
stylegan3_repo = ROOT / "third_party" / "stylegan3"
train_py = stylegan3_repo / "train.py"
loss_py = stylegan3_repo / "training" / "loss.py"
loss_backup = stylegan3_repo / "training" / "loss_before_freq_ablation_backup.py"

stats_npz = ROOT / "data" / "manifests" / "real_landmark_stats.npz"
reg_ckpt = ROOT / "checkpoints" / "aux_models" / "landmark_feature_regressor.pt"

freq_stats_npz = ROOT / "data" / "manifests" / "real_frequency_stats.npz"
freq_stats_json = ROOT / "data" / "manifests" / "real_frequency_stats_summary.json"

baseline_root = ROOT / "checkpoints" / "baseline_stylegan3r"
landmark_root = ROOT / "checkpoints" / "landmark_reg"

if EXPERIMENT_MODE == "freq_only":
    ablation_root = ROOT / "checkpoints" / "freq_ablation"
    logs_dir = ROOT / "logs" / "freq_ablation"
else:
    ablation_root = ROOT / "checkpoints" / "combined_ablation"
    logs_dir = ROOT / "logs" / "combined_ablation"

ablation_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

cache_dir = ROOT / ".cache"
(cache_dir / "torch_extensions").mkdir(parents=True, exist_ok=True)
(cache_dir / "dnnlib").mkdir(parents=True, exist_ok=True)

os.environ["TORCH_EXTENSIONS_DIR"] = str(cache_dir / "torch_extensions")
os.environ["DNNLIB_CACHE_DIR"] = str(cache_dir / "dnnlib")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"

# =========================================================
# PRECHECKS
# =========================================================
assert dataset_zip.exists(), f"Missing dataset ZIP: {dataset_zip}"
assert stylegan3_repo.exists(), f"Missing StyleGAN3 repo: {stylegan3_repo}"
assert train_py.exists(), f"Missing train.py: {train_py}"
assert loss_py.exists(), f"Missing training/loss.py: {loss_py}"

if EXPERIMENT_MODE == "combined":
    assert stats_npz.exists(), f"Missing landmark stats NPZ: {stats_npz}"
    assert reg_ckpt.exists(), f"Missing landmark regressor ckpt: {reg_ckpt}"

# =========================================================
# HELPERS
# =========================================================
def latest_run_dir(root: Path):
    runs = [p for p in root.iterdir() if p.is_dir()] if root.exists() else []
    runs = sorted(runs, key=lambda p: p.stat().st_mtime, reverse=True)
    return runs[0] if runs else None

def latest_snapshot(run_dir: Path):
    if run_dir is None or not run_dir.exists():
        return None
    snaps = sorted(run_dir.glob("network-snapshot-*.pkl"))
    return snaps[-1] if snaps else None

def latest_fakes(run_dir: Path):
    if run_dir is None or not run_dir.exists():
        return None
    imgs = sorted(run_dir.glob("fakes*.png"))
    return imgs[-1] if imgs else None

def tail_jsonl(path: Path, n=5):
    if not path.exists():
        return []
    lines = path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for line in lines[-n:]:
        try:
            out.append(json.loads(line))
        except Exception:
            out.append({"raw": line})
    return out

def run_cmd(cmd, cwd=None, capture=False, env=None):
    if capture:
        return subprocess.check_output(cmd, cwd=cwd, text=True, env=env)
    print("\n[RUN]", " ".join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=cwd, check=True, env=env)

def stream_process(cmd, cwd=None, logfile=None, env=None):
    print("\n[RUN]")
    print(" ".join(str(x) for x in cmd))
    with subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
        env=env,
    ) as proc:
        with open(logfile, "a", encoding="utf-8") if logfile else open(os.devnull, "w") as logf:
            for line in proc.stdout:
                print(line, end="")
                if logfile:
                    logf.write(line)
            proc.wait()
            if proc.returncode != 0:
                raise subprocess.CalledProcessError(proc.returncode, cmd)

def verify_ref_patch():
    files = {
        "bias_act": stylegan3_repo / "torch_utils" / "ops" / "bias_act.py",
        "upfirdn2d": stylegan3_repo / "torch_utils" / "ops" / "upfirdn2d.py",
        "filtered_lrelu": stylegan3_repo / "torch_utils" / "ops" / "filtered_lrelu.py",
    }
    checks = {}
    for k, p in files.items():
        text = p.read_text(encoding="utf-8", errors="ignore")
        checks[k] = "impl='ref'" in text
    return checks

def list_images_in_zip(zip_path: Path):
    IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(IMAGE_EXTS)]
    return sorted(names)

def load_image_from_zip(zip_path: Path, inner_name: str):
    with zipfile.ZipFile(zip_path, "r") as zf:
        data = zf.read(inner_name)
    img = Image.open(io.BytesIO(data)).convert("RGB")
    return np.array(img)

# =========================================================
# COMPUTE REAL FREQUENCY STATS
# =========================================================
def high_freq_ratio_np(img_np, cutoff_ratio=0.25):
    x = img_np.astype(np.float32) / 255.0
    gray = x.mean(axis=2)
    fft = np.fft.fftshift(np.fft.fft2(gray))
    power = np.abs(fft) ** 2

    h, w = power.shape
    yy, xx = np.mgrid[0:h, 0:w]
    cy, cx = h // 2, w // 2
    rr = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    max_r = rr.max() + 1e-8
    mask = rr >= (cutoff_ratio * max_r)

    ratio = float(power[mask].sum() / (power.sum() + 1e-8))
    return ratio

if (not freq_stats_npz.exists()):
    all_imgs = list_images_in_zip(dataset_zip)
    random.seed(AB_SEED)
    sample_n = min(FREQ_SAMPLE_SIZE, len(all_imgs))
    sample_imgs = random.sample(all_imgs, sample_n)

    ratios = []
    for inner_name in tqdm(sample_imgs, desc="Computing real frequency stats"):
        try:
            img_np = load_image_from_zip(dataset_zip, inner_name)
            ratios.append(high_freq_ratio_np(img_np, cutoff_ratio=FREQ_CUTOFF_RATIO))
        except Exception:
            pass

    ratios = np.asarray(ratios, dtype=np.float32)
    assert len(ratios) > 0, "No frequency ratios computed."

    freq_mean = float(ratios.mean())
    freq_std = float(ratios.std() + 1e-8)

    np.savez(
        freq_stats_npz,
        cutoff_ratio=np.float32(FREQ_CUTOFF_RATIO),
        mean=np.float32(freq_mean),
        std=np.float32(freq_std),
        ratios=ratios.astype(np.float32),
    )

    freq_summary = {
        "time": datetime.now().isoformat(),
        "dataset_zip": str(dataset_zip),
        "sample_size": int(len(ratios)),
        "cutoff_ratio": float(FREQ_CUTOFF_RATIO),
        "mean": freq_mean,
        "std": freq_std,
        "stats_npz": str(freq_stats_npz),
    }
    freq_stats_json.write_text(json.dumps(freq_summary, indent=2), encoding="utf-8")
    print("Saved frequency stats:", json.dumps(freq_summary, indent=2))
else:
    print("Using existing frequency stats:", freq_stats_npz)

freq_npz = np.load(freq_stats_npz, allow_pickle=True)
FREQ_TARGET_MEAN = float(freq_npz["mean"])
FREQ_TARGET_STD = float(freq_npz["std"])
print("Frequency target mean:", FREQ_TARGET_MEAN)
print("Frequency target std :", FREQ_TARGET_STD)

# =========================================================
# PATCH STYLEGAN3 LOSS.PY FOR FREQ / COMBINED ABLATION
# =========================================================
if not loss_backup.exists():
    shutil.copy2(loss_py, loss_backup)
    print("Backed up current loss.py to:", loss_backup)
else:
    print("Backup already exists:", loss_backup)

patched_loss_code = r'''
# Copyright (c) 2021, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
"""Loss functions with optional landmark + frequency regularization."""
import os
import numpy as np
import torch
import torch.nn as nn
from torch_utils import training_stats
from torch_utils.ops import conv2d_gradfix
from torch_utils.ops import upfirdn2d

#----------------------------------------------------------------------------

class Loss:
    def accumulate_gradients(self, phase, real_img, real_c, gen_z, gen_c, gain, cur_nimg):
        raise NotImplementedError()

#----------------------------------------------------------------------------

class LandmarkFeatureRegressor(nn.Module):
    def __init__(self, n_outputs):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Linear(128, n_outputs),
        )

    def forward(self, x):
        x = self.net(x)
        x = self.head(x)
        return x

#----------------------------------------------------------------------------

class StyleGAN2Loss(Loss):
    def __init__(self, device, G, D, augment_pipe=None, r1_gamma=10, style_mixing_prob=0, pl_weight=0, pl_batch_shrink=2, pl_decay=0.01, pl_no_weight_grad=False, blur_init_sigma=0, blur_fade_kimg=0):
        super().__init__()
        self.device = device
        self.G = G
        self.D = D
        self.augment_pipe = augment_pipe
        self.r1_gamma = r1_gamma
        self.style_mixing_prob = style_mixing_prob
        self.pl_weight = pl_weight
        self.pl_batch_shrink = pl_batch_shrink
        self.pl_decay = pl_decay
        self.pl_no_weight_grad = pl_no_weight_grad
        self.pl_mean = torch.zeros([], device=device)
        self.blur_init_sigma = blur_init_sigma
        self.blur_fade_kimg = blur_fade_kimg

        # Landmark regularization
        self.lm_reg_enable = os.environ.get("LM_REG_ENABLE", "0") == "1"
        self.lm_reg_lambda = float(os.environ.get("LM_REG_LAMBDA", "0.0"))
        self.lm_reg_z_margin = float(os.environ.get("LM_REG_Z_MARGIN", "2.5"))
        self.lm_regressor = None

        if self.lm_reg_enable:
            ckpt_path = os.environ.get("LM_REG_CKPT", "")
            if ckpt_path and os.path.isfile(ckpt_path):
                ckpt = torch.load(ckpt_path, map_location=device)
                n_outputs = len(ckpt["feature_names"])
                self.lm_regressor = LandmarkFeatureRegressor(n_outputs=n_outputs).to(device)
                self.lm_regressor.load_state_dict(ckpt["model_state_dict"], strict=True)
                self.lm_regressor.eval()
                for p in self.lm_regressor.parameters():
                    p.requires_grad = False
            else:
                self.lm_reg_enable = False

        # Frequency regularization
        self.freq_reg_enable = os.environ.get("FREQ_REG_ENABLE", "0") == "1"
        self.freq_reg_lambda = float(os.environ.get("FREQ_REG_LAMBDA", "0.0"))
        self.freq_cutoff_ratio = float(os.environ.get("FREQ_CUTOFF_RATIO", "0.25"))
        self.freq_target_mean = float(os.environ.get("FREQ_TARGET_MEAN", "0.0"))
        self.freq_target_std = float(os.environ.get("FREQ_TARGET_STD", "1.0"))
        self.freq_z_margin = float(os.environ.get("FREQ_Z_MARGIN", "1.0"))

    def run_G(self, z, c, update_emas=False):
        ws = self.G.mapping(z, c, update_emas=update_emas)
        if self.style_mixing_prob > 0:
            with torch.autograd.profiler.record_function('style_mixing'):
                cutoff = torch.empty([], dtype=torch.int64, device=ws.device).random_(1, ws.shape[1])
                cutoff = torch.where(torch.rand([], device=ws.device) < self.style_mixing_prob, cutoff, torch.full_like(cutoff, ws.shape[1]))
                ws[:, cutoff:] = self.G.mapping(torch.randn_like(z), c, update_emas=False)[:, cutoff:]
        img = self.G.synthesis(ws, update_emas=update_emas)
        return img, ws

    def run_D(self, img, c, blur_sigma=0, update_emas=False):
        blur_size = np.floor(blur_sigma * 3)
        if blur_size > 0:
            with torch.autograd.profiler.record_function('blur'):
                f = torch.arange(-blur_size, blur_size + 1, device=img.device).div(blur_sigma).square().neg().exp2()
                img = upfirdn2d.filter2d(img, f / f.sum())
        if self.augment_pipe is not None:
            img = self.augment_pipe(img)
        logits = self.D(img, c, update_emas=update_emas)
        return logits

    def run_landmark_reg(self, img):
        if (not self.lm_reg_enable) or (self.lm_regressor is None):
            return torch.zeros([], device=img.device)

        x = ((img + 1.0) * 0.5).clamp(0, 1)
        pred_z = self.lm_regressor(x)
        penalty = torch.relu(torch.abs(pred_z) - self.lm_reg_z_margin).square().mean(dim=1)
        training_stats.report('Loss/G/landmark_reg_raw', penalty)
        return penalty

    def run_frequency_reg(self, img):
        if not self.freq_reg_enable:
            return torch.zeros([], device=img.device)

        x = ((img + 1.0) * 0.5).clamp(0, 1)
        gray = x.mean(dim=1)  # [B,H,W]

        fft = torch.fft.fftshift(torch.fft.fft2(gray, dim=(-2, -1)), dim=(-2, -1))
        power = torch.abs(fft) ** 2

        B, H, W = power.shape
        yy = torch.arange(H, device=img.device).view(H, 1).float()
        xx = torch.arange(W, device=img.device).view(1, W).float()
        cy = float(H // 2)
        cx = float(W // 2)
        rr = torch.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
        rr = rr / (rr.max() + 1e-8)

        mask = (rr >= self.freq_cutoff_ratio).float()  # [H,W]
        high = (power * mask.unsqueeze(0)).sum(dim=(1,2))
        total = power.sum(dim=(1,2)) + 1e-8
        ratio = high / total

        z = (ratio - self.freq_target_mean) / (self.freq_target_std + 1e-8)
        penalty = torch.relu(torch.abs(z) - self.freq_z_margin).square()

        training_stats.report('Loss/G/frequency_ratio', ratio)
        training_stats.report('Loss/G/frequency_reg_raw', penalty)
        return penalty

    def accumulate_gradients(self, phase, real_img, real_c, gen_z, gen_c, gain, cur_nimg):
        assert phase in ['Gmain', 'Greg', 'Gboth', 'Dmain', 'Dreg', 'Dboth']
        if self.pl_weight == 0:
            phase = {'Greg': 'none', 'Gboth': 'Gmain'}.get(phase, phase)
        if self.r1_gamma == 0:
            phase = {'Dreg': 'none', 'Dboth': 'Dmain'}.get(phase, phase)
        blur_sigma = max(1 - cur_nimg / (self.blur_fade_kimg * 1e3), 0) * self.blur_init_sigma if self.blur_fade_kimg > 0 else 0

        # Gmain
        if phase in ['Gmain', 'Gboth']:
            with torch.autograd.profiler.record_function('Gmain_forward'):
                gen_img, _gen_ws = self.run_G(gen_z, gen_c)
                gen_logits = self.run_D(gen_img, gen_c, blur_sigma=blur_sigma)
                training_stats.report('Loss/scores/fake', gen_logits)
                training_stats.report('Loss/signs/fake', gen_logits.sign())

                loss_Gadv = torch.nn.functional.softplus(-gen_logits)
                lm_penalty = self.run_landmark_reg(gen_img)
                freq_penalty = self.run_frequency_reg(gen_img)

                loss_Gmain = loss_Gadv
                if self.lm_reg_enable and self.lm_reg_lambda > 0:
                    loss_Gmain = loss_Gmain + self.lm_reg_lambda * lm_penalty
                if self.freq_reg_enable and self.freq_reg_lambda > 0:
                    loss_Gmain = loss_Gmain + self.freq_reg_lambda * freq_penalty

                training_stats.report('Loss/G/adv', loss_Gadv)
                training_stats.report('Loss/G/landmark_reg', lm_penalty)
                training_stats.report('Loss/G/frequency_reg', freq_penalty)
                training_stats.report('Loss/G/loss', loss_Gmain)

            with torch.autograd.profiler.record_function('Gmain_backward'):
                loss_Gmain.mean().mul(gain).backward()

        # Gpl
        if phase in ['Greg', 'Gboth']:
            with torch.autograd.profiler.record_function('Gpl_forward'):
                batch_size = gen_z.shape[0] // self.pl_batch_shrink
                gen_img, gen_ws = self.run_G(gen_z[:batch_size], gen_c[:batch_size])
                pl_noise = torch.randn_like(gen_img) / np.sqrt(gen_img.shape[2] * gen_img.shape[3])
                with torch.autograd.profiler.record_function('pl_grads'), conv2d_gradfix.no_weight_gradients(self.pl_no_weight_grad):
                    pl_grads = torch.autograd.grad(outputs=[(gen_img * pl_noise).sum()], inputs=[gen_ws], create_graph=True, only_inputs=True)[0]
                pl_lengths = pl_grads.square().sum(2).mean(1).sqrt()
                pl_mean = self.pl_mean.lerp(pl_lengths.mean(), self.pl_decay)
                self.pl_mean.copy_(pl_mean.detach())
                pl_penalty = (pl_lengths - pl_mean).square()
                training_stats.report('Loss/pl_penalty', pl_penalty)
                loss_Gpl = pl_penalty * self.pl_weight
                training_stats.report('Loss/G/reg', loss_Gpl)
            with torch.autograd.profiler.record_function('Gpl_backward'):
                loss_Gpl.mean().mul(gain).backward()

        # Dmain on fakes
        loss_Dgen = 0
        if phase in ['Dmain', 'Dboth']:
            with torch.autograd.profiler.record_function('Dgen_forward'):
                gen_img, _gen_ws = self.run_G(gen_z, gen_c, update_emas=True)
                gen_logits = self.run_D(gen_img, gen_c, blur_sigma=blur_sigma, update_emas=True)
                training_stats.report('Loss/scores/fake', gen_logits)
                training_stats.report('Loss/signs/fake', gen_logits.sign())
                loss_Dgen = torch.nn.functional.softplus(gen_logits)
            with torch.autograd.profiler.record_function('Dgen_backward'):
                loss_Dgen.mean().mul(gain).backward()

        # Dmain / Dr1 on reals
        if phase in ['Dmain', 'Dreg', 'Dboth']:
            name = 'Dreal' if phase == 'Dmain' else 'Dr1' if phase == 'Dreg' else 'Dreal_Dr1'
            with torch.autograd.profiler.record_function(name + '_forward'):
                real_img_tmp = real_img.detach().requires_grad_(phase in ['Dreg', 'Dboth'])
                real_logits = self.run_D(real_img_tmp, real_c, blur_sigma=blur_sigma)
                training_stats.report('Loss/scores/real', real_logits)
                training_stats.report('Loss/signs/real', real_logits.sign())

                loss_Dreal = 0
                if phase in ['Dmain', 'Dboth']:
                    loss_Dreal = torch.nn.functional.softplus(-real_logits)
                    training_stats.report('Loss/D/loss', loss_Dgen + loss_Dreal)

                loss_Dr1 = 0
                if phase in ['Dreg', 'Dboth']:
                    with torch.autograd.profiler.record_function('r1_grads'), conv2d_gradfix.no_weight_gradients():
                        r1_grads = torch.autograd.grad(outputs=[real_logits.sum()], inputs=[real_img_tmp], create_graph=True, only_inputs=True)[0]
                    r1_penalty = r1_grads.square().sum([1,2,3])
                    loss_Dr1 = r1_penalty * (self.r1_gamma / 2)
                    training_stats.report('Loss/r1_penalty', r1_penalty)
                    training_stats.report('Loss/D/reg', loss_Dr1)

            with torch.autograd.profiler.record_function(name + '_backward'):
                (loss_Dreal + loss_Dr1).mean().mul(gain).backward()

#----------------------------------------------------------------------------
'''.strip()

loss_py.write_text(patched_loss_code + "\n", encoding="utf-8")
print("Patched loss.py for frequency ablation:", loss_py)

# =========================================================
# CHOOSE RESUME SOURCE
# =========================================================
baseline_run = latest_run_dir(baseline_root)
baseline_snapshot = latest_snapshot(baseline_run) if baseline_run is not None else None
landmark_run = latest_run_dir(landmark_root)
landmark_snapshot = latest_snapshot(landmark_run) if landmark_run is not None else None

if EXPERIMENT_MODE == "combined":
    # safest today: resume from baseline, then enable both losses
    resume_source = baseline_snapshot if baseline_snapshot is not None else PRETRAINED_URL
    resume_mode = "resume_from_baseline_for_combined"
else:
    resume_source = baseline_snapshot if baseline_snapshot is not None else PRETRAINED_URL
    resume_mode = "resume_from_baseline_for_freq_only"

assert resume_source is not None, "Could not determine resume source."

# =========================================================
# BUILD TRAIN CMD
# =========================================================
def build_train_cmd(resume_source):
    return [
        sys.executable, str(train_py),
        "--outdir", str(ablation_root),
        "--cfg", CFG,
        "--data", str(dataset_zip),
        "--gpus", "1",
        "--batch", str(BATCH),
        "--batch-gpu", str(BATCH_GPU),
        "--gamma", str(GAMMA),
        "--mirror", str(MIRROR),
        "--kimg", str(AB_KIMG),
        "--tick", str(AB_TICK),
        "--snap", str(AB_SNAP),
        "--metrics", str(AB_METRICS),
        "--seed", str(AB_SEED),
        "--resume", str(resume_source),
        "--aug", str(AUG),
        "--cbase", str(CBASE),
        "--glr", str(GLR),
        "--dlr", str(DLR),
        "--workers", str(WORKERS),
        "--mbstd-group", str(MBSTD_GROUP),
    ]

env = os.environ.copy()
env["FREQ_REG_ENABLE"] = "1"
env["FREQ_REG_LAMBDA"] = str(FREQ_LAMBDA)
env["FREQ_CUTOFF_RATIO"] = str(FREQ_CUTOFF_RATIO)
env["FREQ_TARGET_MEAN"] = str(FREQ_TARGET_MEAN)
env["FREQ_TARGET_STD"] = str(FREQ_TARGET_STD)
env["FREQ_Z_MARGIN"] = str(FREQ_Z_MARGIN)

if EXPERIMENT_MODE == "combined":
    env["LM_REG_ENABLE"] = "1"
    env["LM_REG_CKPT"] = str(reg_ckpt)
    env["LM_REG_LAMBDA"] = str(LM_LAMBDA)
    env["LM_REG_Z_MARGIN"] = str(LM_Z_MARGIN)
else:
    env["LM_REG_ENABLE"] = "0"

cmd = build_train_cmd(resume_source)

# =========================================================
# STATUS MODE
# =========================================================
if MODE.strip().lower() == "status":
    run_dir = latest_run_dir(ablation_root)
    print("Latest ablation run:", run_dir)
    if run_dir is None:
        print("No ablation run found yet.")
    else:
        snap = latest_snapshot(run_dir)
        fake = latest_fakes(run_dir)
        stats_file = run_dir / "training_stats.jsonl"
        print("Latest snapshot:", snap)
        print("Latest fake grid:", fake)
        print("Stats file exists:", stats_file.exists())
        if stats_file.exists():
            print("\nLast few stat entries:")
            for row in tail_jsonl(stats_file, n=5):
                print(json.dumps(row, indent=2))

# =========================================================
# TRAIN MODE
# =========================================================
elif MODE.strip().lower() == "train":
    launch_info = {
        "time": datetime.now().isoformat(),
        "experiment_mode": EXPERIMENT_MODE,
        "resume_mode": resume_mode,
        "resume_source": str(resume_source),
        "dataset_zip": str(dataset_zip),
        "freq_stats_npz": str(freq_stats_npz),
        "freq_lambda": FREQ_LAMBDA,
        "freq_cutoff_ratio": FREQ_CUTOFF_RATIO,
        "freq_target_mean": FREQ_TARGET_MEAN,
        "freq_target_std": FREQ_TARGET_STD,
        "freq_z_margin": FREQ_Z_MARGIN,
        "lm_enabled": EXPERIMENT_MODE == "combined",
        "lm_lambda": LM_LAMBDA if EXPERIMENT_MODE == "combined" else 0.0,
        "cfg": CFG,
        "batch": BATCH,
        "batch_gpu": BATCH_GPU,
        "gamma": GAMMA,
        "cbase": CBASE,
        "glr": GLR,
        "dlr": DLR,
        "aug": AUG,
        "mirror": MIRROR,
        "workers": WORKERS,
        "mbstd_group": MBSTD_GROUP,
        "ab_kimg": AB_KIMG,
        "ab_tick": AB_TICK,
        "ab_snap": AB_SNAP,
        "ab_metrics": AB_METRICS,
        "seed": AB_SEED,
        "command": " ".join(str(x) for x in cmd),
    }

    launch_json = logs_dir / f"{EXPERIMENT_MODE}_launcher_config.json"
    launch_json.write_text(json.dumps(launch_info, indent=2), encoding="utf-8")
    print("Saved launcher config:", launch_json)
    print(json.dumps(launch_info, indent=2))

    live_log = logs_dir / f"{EXPERIMENT_MODE}_train_live.log"
    stream_process(cmd, cwd=stylegan3_repo, logfile=live_log, env=env)

    run_dir = latest_run_dir(ablation_root)
    snap = latest_snapshot(run_dir) if run_dir else None
    fake = latest_fakes(run_dir) if run_dir else None

    print("\nAblation training finished.")
    print("Run dir:", run_dir)
    print("Latest snapshot:", snap)
    print("Latest fake grid:", fake)
    print("Live log:", live_log)

else:
    raise ValueError("MODE must be 'train' or 'status'")

print("\nNotebook 08 complete.")
print("Next notebook: 09_metrics_and_ablation_analysis.ipynb")

Computing real frequency stats:   0%|          | 0/3000 [00:00<?, ?it/s]

Saved frequency stats: {
  "time": "2026-04-09T18:52:05.152945",
  "dataset_zip": "/data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/data/stylegan3_zip/celeba_256x256.zip",
  "sample_size": 3000,
  "cutoff_ratio": 0.25,
  "mean": 0.003470663446933031,
  "std": 0.003451707528362274,
  "stats_npz": "/data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/data/manifests/real_frequency_stats.npz"
}
Frequency target mean: 0.003470663446933031
Frequency target std : 0.0034517075400799513
Backed up current loss.py to: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/third_party/stylegan3/training/loss_before_freq_ablation_backup.py
Patched loss.py for frequency ablation: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/third_party/stylegan3/training/loss.py
Saved launcher config: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/logs/freq_ablation/freq_only_launcher_config.json
{
  "time": "2026-04-09T18:52:05.156058",
  "experiment_mode